# Anti-DPO dataset audit

This standalone experiment maps source `rejected` to DPO `chosen` and uses a bounded penalty for verbose source `chosen` completions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT.parent / 'train.jsonl'
assert SOURCE.is_file(), SOURCE

In [ ]:
from anti_preference import load_jsonl_pairs, invert_preference_pairs

rows, report = load_jsonl_pairs(SOURCE)
anti_rows = invert_preference_pairs(rows, penalty_strength=.35, max_weight=2.0)
report.to_dict(), anti_rows[0]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist([len(row['chosen']) for row in rows], bins=60, range=(0, 2500), alpha=.7, label='source chosen')
axes[0].hist([len(row['rejected']) for row in rows], bins=60, range=(0, 2500), alpha=.7, label='source rejected')
axes[0].set(xlabel='Characters', ylabel='Rows', title='Response lengths')
axes[0].legend()
axes[1].hist([row['anti_weight'] for row in anti_rows], bins=30, color='#9c2c77')
axes[1].set(xlabel='Multiplier', ylabel='Rows', title='Bounded anti-DPO weight')
fig.tight_layout()